# 从 OpenAI Agents SDK 迁移

如果你的应用建立在 OpenAI Agents SDK 上并希望移植到 Claude Agent SDK，本笔记本用一个统一的示例——费用审批智能体——逐个对应地说明每个原语。

**迁移后你得到什么。** Claude Agent SDK 运行在与 Claude Code 相同的运行时上——你继承它内置的 `Read`、`Edit`、`Bash` 和 `Grep` 工具、一套分层的权限系统用于限制智能体能触碰什么、自动提示词缓存，以及对事件流的直接访问，可用于进度流式输出或运行中途拦截。工具定义是显式的（你自己声明模式，而不是依赖类型注解自省），循环由你自己驱动。多数移植是每个工具多一点样板代码，而其他地方少一点样板代码。

## 学完本笔记本，你将能够：

- 用 Claude 的对应物替换 `@function_tool`、护栏和 `Runner.run`，而不重写你的业务逻辑
- 移植单智能体应用：自定义工具、输入/输出护栏、多轮会话和可持久化的恢复
- 知道何时该用 `ClaudeSDKClient`，何时该用无状态的 `query()` 函数
- 把 SDK 的 OpenTelemetry 导出接到你现有的可观测性栈上

两个 SDK 都会实际运行。你需要在 `.env` 中设置 `OPENAI_API_KEY` 和 `ANTHROPIC_API_KEY`。

| OpenAI Agents SDK | Claude Agent SDK |
|---|---|
| `Agent(name, instructions, tools)` | `ClaudeAgentOptions` + 系统提示词 |
| `@function_tool` | `@tool` + `create_sdk_mcp_server` |
| `@input_guardrail` | 循环前的普通函数（或 `UserPromptSubmit` 钩子） |
| `@output_guardrail` | 作用于 `ResultMessage.result` 的普通函数 |
| `Runner.run(agent, msg)` | `ClaudeSDKClient` 上下文管理器 |
| `Sessions`（客户端管理） | 复用同一个 `ClaudeSDKClient` |
| `conversation_id`（服务端管理） | `resume=session_id`（磁盘存储） |
| 内置跟踪看板 | OTel 原生——接入你现有的 Grafana/Datadog/Honeycomb |
| `handoffs=[...]` | `AgentDefinition` + Agent 工具——见附录 |

## 示例：费用审批

一个智能体，负责批准或标记费用提交。一个工具（`check_policy`），一个输入护栏（没有金额就拒绝），加上循环。

我们先用 OpenAI Agents SDK 构建一次，然后逐个移植每个原语。

## 前置条件

**需要掌握的知识**

- 熟悉 Python 的 `async`/`await`
- 熟悉你要从中迁移过来的 OpenAI Agents SDK 原语

**需要的工具**

- Python 3.11 或 3.12
- `ANTHROPIC_API_KEY`（[点此获取](https://console.anthropic.com)）和 `OPENAI_API_KEY`（[点此获取](https://platform.openai.com/api-keys)）——两个 SDK 都会实际运行

**建议：**

- 略读[笔记本 00](./00_The_one_liner_research_agent.ipynb)，了解 `query()` 和 `ClaudeSDKClient` 的基础

> `openai-agents` 固定在 `0.9.3`——它尚未到 1.0，API 经常变化。如果你升级版本，请重新核对 `Agent`/`Runner`/`@function_tool` 的签名。

In [1]:
%%capture
# openai-agents pinned to 0.9.3 — see prereqs note before bumping
%pip install "openai-agents==0.9.3" claude-agent-sdk python-dotenv

In [2]:
import os
import re
from dataclasses import replace

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY missing from .env"

OAI_MODEL = "gpt-4.1"
CLAUDE_MODEL = "claude-sonnet-4-6"

In [3]:
# ===== OpenAI Agents SDK — full implementation =====
from agents import Agent, GuardrailFunctionOutput, Runner, function_tool, input_guardrail


@function_tool
def check_policy(category: str, amount: float) -> dict:
    """Look up the expense policy for a category. Returns the approval limit. Valid categories: meals, travel, software, other."""
    limits = {"meals": 75.0, "travel": 500.0, "software": 200.0, "other": 50.0}
    return {
        "category": category,
        "limit": limits.get(category.lower(), 50.0),
        "requires_receipt": amount > 25.0,
    }


@input_guardrail
def has_dollar_amount(ctx, agent, user_input: str) -> GuardrailFunctionOutput:
    has_amount = bool(re.search(r"\$\d+", user_input))
    return GuardrailFunctionOutput(
        output_info={"has_amount": has_amount},
        tripwire_triggered=not has_amount,
    )


expense_agent = Agent(
    name="Expense Approver",
    instructions=(
        "You approve or flag expense submissions. "
        "Always call check_policy first to get the limit for the expense category. "
        "Approve if the amount is under the limit; otherwise flag for manager review."
    ),
    tools=[check_policy],
    input_guardrails=[has_dollar_amount],
    model=OAI_MODEL,
)


async def run_oai(msg: str) -> str:
    result = await Runner.run(expense_agent, msg)
    return result.final_output


print(await run_oai("Lunch with Acme, $47"))

This lunch expense of $47 falls under the "meals" category, which has an approval limit of $75. Since the amount is below the limit, I approve the expense (assuming a receipt is provided, as required by policy).


## 逐个原语地移植

### `@function_tool` → `@tool` + `create_sdk_mcp_server`

**OpenAI：** `@function_tool` 从类型注解和文档字符串推导工具模式。被装饰的函数直接放进 `tools=[...]`。

**Claude：** `@tool` 把名称、描述和模式作为显式参数——你写什么，模型就看到什么。处理函数是 `async` 的，接收单个 `args` 字典，并返回 `{"content": [{"type": "text", "text": ...}]}`。工具被打包进一个进程内 MCP 服务器（尽管名字如此，但没有子进程或网络传输），这才是你传给智能体的单元。

你函数内部的逻辑不变。变的只是外层包装。

In [4]:
import json

from claude_agent_sdk import create_sdk_mcp_server, tool


@tool(
    "check_policy",
    "Look up the expense policy for a category. Returns the approval limit. Valid categories: meals, travel, software, other.",
    {"category": str, "amount": float},
)
async def check_policy_claude(args):
    limits = {"meals": 75.0, "travel": 500.0, "software": 200.0, "other": 50.0}
    category, amount = args["category"], args["amount"]
    result = {
        "category": category,
        "limit": limits.get(category.lower(), 50.0),
        "requires_receipt": amount > 25.0,
    }
    return {"content": [{"type": "text", "text": json.dumps(result)}]}


policy_server = create_sdk_mcp_server(name="expense", tools=[check_policy_claude])

### `Agent(...)` → `ClaudeAgentOptions`

`ClaudeAgentOptions` 承载模型、工具和系统提示词——也就是实际发送给 Claude 的配置。你的护栏逻辑留在应用代码中，而不是注册在框架对象上。OpenAI 的 `Agent` 把这些全部打包在一起。

| `Agent(...)` 字段 | `ClaudeAgentOptions` 对应项 |
|---|---|
| `instructions=...` | `system_prompt=...` |
| `tools=[fn]` | `mcp_servers={...}` + `allowed_tools=[...]` |
| `model=...` | `model=...` |
| `name=...` | 不需要——options 是一个配置值，不是具名实体。子智能体名称写在 `AgentDefinition` 上（见附录）。|
| `input_guardrails=[...]` | 在你的运行函数中调用——见下一节 |

`allowed_tools` 中的工具名称遵循 `mcp__{server_name}__{tool_name}` 的形式。

> **关于权限的说明：** `allowed_tools` 是一条允许规则——它使工具*对*智能体*可用*。智能体能否不经用户批准就调用它，取决于 `permission_mode`。像 `check_policy` 这样的只读自定义工具默认自由运行；会写文件或执行 shell 命令的工具会提示，除非你设置 `permission_mode="acceptEdits"` 或 `"bypassPermissions"`。如果你的工具不止是读取，请看[权限指南](https://platform.claude.com/docs/en/agent-sdk/permissions)。

In [5]:
from claude_agent_sdk import ClaudeAgentOptions

expense_system_prompt = (
    "You approve or flag expense submissions. "
    "Always call check_policy first to get the limit for the expense category. "
    "Approve if the amount is under the limit; otherwise flag for manager review."
)

expense_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt=expense_system_prompt,
    mcp_servers={"expense": policy_server},
    allowed_tools=["mcp__expense__check_policy"],
)

> **内置工具。** 费用智能体只用到一个自定义工具，但 SDK 提供了 `Read`、`Edit`、`Bash`、`Grep` 等——按名称把它们加入 `allowed_tools`（例如 `allowed_tools=["Read", "Grep", "mcp__expense__check_policy"]`）。文件系统用法见[笔记本 00](./00_The_one_liner_research_agent.ipynb)。

### `@input_guardrail` / `@output_guardrail` → 调用前/调用后检查

**OpenAI：** `@input_guardrail` 校验用户的消息；`@output_guardrail` 校验智能体的最终答案。两者都通过返回 `GuardrailFunctionOutput(tripwire_triggered=True)` 来阻断，这会抛出 `InputGuardrailTripwireTriggered`（或输出变体）。输入护栏与智能体并发运行。

**Claude：** 你的检查就是普通函数。在启动客户端之前调用输入检查，在循环结束后对 `ResultMessage.result` 调用输出检查。

| OAI | Claude |
|---|---|
| `@input_guardrail` 装饰器 | 循环前调用的普通函数 |
| `@output_guardrail` 装饰器 | 循环后对 `result.result` 调用的普通函数 |
| `tripwire_triggered=True` 抛出异常 | 提前返回（输入）或覆盖结果（输出） |
| 注册在 `Agent(input_guardrails=[...])` 上 | 在你的运行函数中调用 |

#### `UserPromptSubmit` 钩子这一替代方案

SDK 的 `UserPromptSubmit` 钩子在提示词到达 Claude 之前触发，并且可以阻断它——这是与 `@input_guardrail` 结构上最接近的对应物。下面两种都展示，但演示用普通函数，原因有二：拒绝消息由你自己掌控（当钩子阻断时，`ResultMessage.result` 返回为空——原因不会呈现给调用方），并且护栏逻辑留在你的运行函数中可见，而不是注册在 options 对象上。`@output_guardrail` 没有干净的钩子对应物——`Stop` 在响应完成后触发，但不会改写输出。至于用 `PreToolUse` 钩子限制工具使用，见[笔记本 03](./03_The_site_reliability_agent.ipynb)，那是另一种用途。

In [6]:
def has_dollar_amount_check(user_input: str) -> tuple[bool, str | None]:
    """Input check — returns (allowed, rejection_message)."""
    if re.search(r"\$\d+", user_input):
        return True, None
    return False, "I need a dollar amount to process this. Please include one (e.g., '$47')."


def has_decision_check(result: str) -> tuple[bool, str | None]:
    """Output check — returns (allowed, override_message)."""
    # Stem match is intentionally permissive for a demo; tighten for production.
    if re.search(r"\b(approv|flag|review)", result, re.IGNORECASE):
        return True, None
    return False, "I couldn't reach a clear approve/flag decision. Please resubmit."

作为对比，这里把同一个检查写成 `UserPromptSubmit` 钩子。回调接收 `input_data["prompt"]`，返回 `{}` 表示允许，返回 `{"decision": "block", "reason": "..."}` 表示阻断。这里 `HookMatcher` 不接 `matcher` 参数，因为 `UserPromptSubmit` 对每个提示词都触发——没有工具名称可供过滤。

要试用它，请在下一节的 `ClaudeSDKClient(...)` 调用中把 `options=expense_options` 换成 `options=hooked_options`。有一个注意点：当 `UserPromptSubmit` 钩子阻断时，阻断原因目前不会呈现在 `ResultMessage.result` 中——你拿回的是空字符串而不是拒绝文本。这就是演示循环使用普通函数的原因，它自己控制返回值。

至于工具级护栏——守护智能体*做什么*而不是用户*发送什么*——见[笔记本 03](./03_The_site_reliability_agent.ipynb) 中的 `PreToolUse` 钩子模式。

In [7]:
from claude_agent_sdk import HookMatcher


async def has_dollar_amount_hook(input_data, tool_use_id, context):
    # tool_use_id is None for UserPromptSubmit — it's a uniform signature across all hook types.
    if re.search(r"\$\d+", input_data["prompt"]):
        return {}
    return {"decision": "block", "reason": "I need a dollar amount to process this."}


hooked_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt=expense_system_prompt,
    mcp_servers={"expense": policy_server},
    allowed_tools=["mcp__expense__check_policy"],
    hooks={"UserPromptSubmit": [HookMatcher(hooks=[has_dollar_amount_hook])]},
)

### `Runner.run()` → `ClaudeSDKClient`

**OpenAI：** `result = await Runner.run(agent, msg)` 运行循环并返回一个结果对象。读取 `result.final_output` 得到文本。

**Claude：** `ClaudeSDKClient` 是一个异步上下文管理器。调用 `.query(msg)` 发送，然后迭代 `.receive_response()` 按到达顺序处理事件——你会按顺序看到每一次工具调用、每一个文本块和最终结果。事件类型如下：

| 事件 | 包含内容 |
|---|---|
| `SystemMessage` | 会话初始化元数据 |
| `AssistantMessage` | 文本块或工具使用块 |
| `UserMessage` | 工具结果块 |
| `ResultMessage` | 总是最后：`.result`、`.usage`、`.total_cost_usd` |

最终答案就是 `messages[-1].result`。

> SDK 还提供了一个无状态的 `query()` 函数（见[笔记本 00](./00_The_one_liner_research_agent.ipynb)），用于使用内置工具的一次性调用。当你通过 `create_sdk_mcp_server` 引入自定义工具时，`ClaudeSDKClient` 是正确选择，因为它的持久传输层负责处理进程内 MCP 握手。

In [8]:
from utils.agent_visualizer import display_agent_response, print_activity, reset_activity_context

from claude_agent_sdk import ClaudeSDKClient


async def run_claude(msg: str) -> None:
    # Input guardrail — short-circuit before the agent runs
    allowed, rejection = has_dollar_amount_check(msg)
    if not allowed:
        print(rejection)
        return

    reset_activity_context()
    messages = []
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query(msg)
        async for event in client.receive_response():
            print_activity(event)
            messages.append(event)

    # receive_response() guarantees ResultMessage is the last event yielded
    final = messages[-1].result
    ok, override = has_decision_check(final or "")
    if not ok:
        print(override)
        return

    display_agent_response(messages)


await run_claude("Lunch with Acme, $47")

🤖 Thinking...
🤖 Thinking...
🤖 Using: mcp__expense__check_policy()
✓ Tool completed
🤖 Thinking...


### `Sessions` → `ClaudeSDKClient`（内存中）或 `resume=`（磁盘存储）

OpenAI 提供两种会话模式：

| OAI 模式 | 持久性 | Claude 对应项 |
|---|---|---|
| `result.to_input_list()` + 重新发送 | 客户端在内存中保存历史 | 复用同一个 `ClaudeSDKClient`——在同一个打开的上下文中再次调用 `.query()` |
| `conversation_id` | 服务端，进程重启后仍存在 | `resume=session_id`——对话记录写在本地磁盘上，重启后仍存在 |

**内存中（见下）：** 保持 `ClaudeSDKClient` 上下文打开并再次调用 `.query()`。无需重新发送历史。进程结束时会话随之消失。

**磁盘存储：** 每次运行都会产生一个 `session_id`（在 `ResultMessage.session_id` 上）。保存它，然后在下一次运行时传入 `ClaudeAgentOptions(resume=session_id, ...)`。对话记录存放在本地文件系统上，而不是 Anthropic 的服务器上——它在重启之间持久，但在不同机器之间不持久。

在任一模式下，当对话超出上下文窗口时，见 [`../misc/session_memory_compaction.ipynb`](../misc/session_memory_compaction.ipynb)。

In [9]:
# --- In-memory: reuse the same client ---
async def run_claude_multiturn():
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query("Lunch with Acme, $47")
        async for _ in client.receive_response():
            pass  # consume the stream

        # Same client — remembers turn 1
        await client.query("What about $90?")
        turn2 = [m async for m in client.receive_response()]
        # receive_response() guarantees ResultMessage is the last event yielded
        return turn2[-1].result


print(await run_claude_multiturn())

🚩 **Flagged for Manager Review**

| Detail | Info |
|---|---|
| **Description** | Lunch with Acme |
| **Category** | Meals |
| **Amount** | $90.00 |
| **Policy Limit** | $75.00 |
| **Over Limit By** | $15.00 |
| **Status** | **Flagged** |

The $90 lunch exceeds the $75 meals limit by **$15.00**, so it requires manager approval before it can be reimbursed.


In [10]:
# --- Disk-backed: capture session_id, resume later ---

# Turn 1: capture the session_id from the ResultMessage (always the last event)
async with ClaudeSDKClient(options=expense_options) as client:
    await client.query("Lunch with Acme, $47")
    turn1 = [m async for m in client.receive_response()]
session_id = turn1[-1].session_id

# Turn 2: new client, new process — resume from disk
resume_opts = replace(expense_options, resume=session_id)
async with ClaudeSDKClient(options=resume_opts) as client:
    await client.query("What about $90?")
    turn2 = [m async for m in client.receive_response()]
print(f"[resumed {session_id[:8]}...] {turn2[-1].result}")

[resumed 4fc6dc1e...] Here's the rundown for both expenses:

| Expense | Amount | Limit | Status |
|---|---|---|---|
| Lunch with Acme | $47 | $75 | ✅ **Approved** |
| Lunch with Acme | $90 | $75 | 🚩 **Flagged for Manager Review** |

- **$47** – Under the $75 meals limit, so it's **approved**. (Receipt required.)
- **$90** – Exceeds the $75 meals limit by $15, so it's **flagged for manager review**.


## 并排对比

把三个测试输入分别通过两个 SDK 运行。我们预期两者都会批准 47 美元的午餐、标记 650 美元的航班，并拒绝没有金额的输入。

In [11]:
from agents.exceptions import InputGuardrailTripwireTriggered


async def run_claude_quiet(msg: str) -> str:
    # Same guardrail as run_claude — duplicated here for a minimal comparison helper
    allowed, rejection = has_dollar_amount_check(msg)
    if not allowed:
        return rejection
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query(msg)
        messages = [m async for m in client.receive_response()]
        # receive_response() guarantees ResultMessage is the last event yielded
        return messages[-1].result or ""


test_inputs = [
    ("Lunch with Acme, $47", "approve"),
    ("Flight to NYC, $650", "flag"),
    ("Need approval for the thing", "guardrail"),
]

for msg, expected in test_inputs:
    print(f"\n{'=' * 60}\nINPUT: {msg}  (expect: {expected})\n{'=' * 60}")

    try:
        oai_out = await run_oai(msg)
    except InputGuardrailTripwireTriggered:
        oai_out = "[guardrail: InputGuardrailTripwireTriggered]"

    claude_out = await run_claude_quiet(msg)

    print(f"OAI:    {str(oai_out)[:200]}")
    print(f"Claude: {str(claude_out)[:200]}")


INPUT: Lunch with Acme, $47  (expect: approve)
OAI:    The expense for lunch ($47) is under the meals category limit of $75. This expense is approved.
Claude: ✅ **Expense Approved**

| Detail | Info |
|---|---|
| **Description** | Lunch with Acme |
| **Category** | Meals |
| **Amount** | $47.00 |
| **Meal Limit** | $75.00 |
| **Status** | **Approved** |

Yo

INPUT: Flight to NYC, $650  (expect: flag)
OAI:    The approval limit for travel expenses is $500. Since your flight to NYC costs $650, this expense exceeds the limit and will be flagged for manager review.
Claude: 🚩 **Expense Flagged for Manager Review**

| Detail | Info |
|---|---|
| **Description** | Flight to NYC |
| **Category** | Travel |
| **Amount** | $650.00 |
| **Policy Limit** | $500.00 |
| **Over Lim

INPUT: Need approval for the thing  (expect: guardrail)
OAI:    [guardrail: InputGuardrailTripwireTriggered]
Claude: I need a dollar amount to process this. Please include one (e.g., '$47').


## 跟踪

Claude 输出到你团队已经在运行的标准。

**每次运行的指标已经在你手中。** 每个 `ResultMessage` 都带有 `.total_cost_usd` 和 `.usage`——输入/输出词元、缓存读取、缓存写入、轮次计数。你在下面的缓存单元格中会看到这些。要做程序化的成本跟踪，见 [`../observability/usage_cost_api.ipynb`](../observability/usage_cost_api.ipynb)。

**OpenTelemetry 是原生的。** 该 SDK 与 Claude Code 共用运行时，后者在启用时导出 OTel 指标和事件：

```bash
export CLAUDE_CODE_ENABLE_TELEMETRY=1
export OTEL_METRICS_EXPORTER=otlp
export OTEL_LOGS_EXPORTER=otlp
export OTEL_EXPORTER_OTLP_PROTOCOL=grpc
export OTEL_EXPORTER_OTLP_ENDPOINT=http://your-collector:4317
```

这些都不是移植所必需的——无论遥测开关与否，你的智能体都以完全相同的方式运行。

每次工具使用和每个 API 请求都会发出一个带有 `prompt.id` 标记的事件，因此你可以把所有活动关联回发起它的提示词。把它指向你现有的 Grafana / Datadog / Honeycomb / Langfuse 栈。完整事件模式见[监控文档](https://code.claude.com/docs/en/monitoring-usage)。

## Claude 独有：提示词缓存

系统提示词和工具模式在首次调用后自动缓存——无需配置。后续使用相同前缀的调用，该前缀只需支付大约输入词元成本的 10%。

> 由于前面的单元格已经用 `expense_options` 运行过，走到这里时缓存已经是热的——所以即使 Run 1 也显示大量 `cache_read_input_tokens`。在一个冷启动的笔记本中，Run 1 主要是 `cache_creation`，Run 2 主要是 `cache_read`。工具使用循环每次查询会发出多个 API 请求，每个请求都在延长被缓存的前缀——所以即使在重复运行时 `cache_creation` 也不会降到零，但它应当缩小，而 `cache_read` 应当增长。

In [12]:
for i in range(2):
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query("Coffee, $6")
        messages = [m async for m in client.receive_response()]
    # receive_response() guarantees ResultMessage is the last event yielded
    usage = messages[-1].usage or {}
    print(
        f"Run {i + 1}: cache_creation={usage.get('cache_creation_input_tokens', 0):>5}  "
        f"cache_read={usage.get('cache_read_input_tokens', 0):>5}  "
        f"input={usage.get('input_tokens', 0):>5}"
    )

Run 1: cache_creation=38728  cache_read=66272  input=    4
Run 2: cache_creation=  659  cache_read=95025  input=    4


---
## 附录：移植 `handoffs`

如果你的 OpenAI Agents SDK 应用使用了 `handoffs=[...]`，这里就是心智模型转变的地方。

**OpenAI：** `handoffs=[specialist]` 会暴露一个 `transfer_to_specialist` 工具。一旦被调用，专家智能体**接管**对话。第一个智能体就此结束。

**Claude：** 子智能体通过 `AgentDefinition` 以编程方式定义，并通过 `ClaudeAgentOptions(agents={...})` 传入。编排者通过 Agent 工具**委派**，取回结果，并始终保持控制权。

对于纯路由器（分诊 → 专家），差别很小。对于“智能体 B 接管、A 再也不运行”是承重设计的应用，考虑用一层薄薄的 Python 分发器，而不是由 LLM 决定的路由。

> 在 Claude Code CLI 工作流中，你可能也会看到把子智能体定义为 `.claude/agents/*.md` 文件（例如[笔记本 01](./01_The_chief_of_staff_agent.ipynb)）。那是面向交互式使用的基于文件系统的路径。对于以代码形式交付的 SDK 应用，下面编程式的 `agents=` 参数才是自然选择——没有文件系统依赖。

In [13]:
from claude_agent_sdk import AgentDefinition

approver = AgentDefinition(
    description="Approves expenses under policy limits. Use for any submission where the amount is at or under the limit returned by check_policy.",
    prompt="You approve expense submissions that are within policy. Confirm the amount and category, and remind the submitter if a receipt is required.",
    tools=["mcp__expense__check_policy"],
)

escalator = AgentDefinition(
    description="Escalates over-limit expenses to a manager. Use when the submitted amount exceeds the policy limit for its category.",
    prompt="You escalate over-limit expenses. Draft a one-line note to the manager: include the amount, the category, and how far over the limit it is.",
    tools=["mcp__expense__check_policy"],
)

triage_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt="Route each expense to the appropriate subagent. Delegate to approver if under limit, escalator if over.",
    mcp_servers={"expense": policy_server},
    allowed_tools=["Agent", "mcp__expense__check_policy"],
    agents={"approver": approver, "escalator": escalator},
)

print(f"Orchestrator configured with subagents: {list(triage_options.agents)}")

# Config only — not executed here. For a full end-to-end multi-agent run,
# see ./01_The_chief_of_staff_agent.ipynb

Orchestrator configured with subagents: ['approver', 'escalator']


## 结论

### 你构建了什么

一个费用审批智能体，逐个原语地移植：`@function_tool` → `@tool`、`Agent` → `ClaudeAgentOptions`、`@input_guardrail`/`@output_guardrail` → 普通的调用前/调用后检查、`Runner.run` → `ClaudeSDKClient`，以及内存中与磁盘存储两种会话。

### 关键要点

- **工具：** 显式模式意味着你写什么，模型就看到什么——没有自省带来的意外。打包为进程内 MCP 服务器，并通过 `ClaudeSDKClient` 传入。
- **护栏：** 普通函数把拒绝流程放在你的代码里，而不是某个框架里。当你希望把它注册在 options 上时，`UserPromptSubmit` 钩子可供使用。
- **会话：** 内存中的多轮用同一个 `ClaudeSDKClient`。`resume=session_id` 用于重启后仍存在的磁盘存储对话。
- **跟踪：** OpenTelemetry 原生，因此可以接入你已经在运行的栈。每轮成本和词元用量都在每个 `ResultMessage` 上——无需额外接线。

### 下一步

- 多智能体编排：[01_The_chief_of_staff_agent.ipynb](./01_The_chief_of_staff_agent.ipynb)
- 工具生命周期钩子（`PreToolUse`/`PostToolUse`）：[03_The_site_reliability_agent.ipynb](./03_The_site_reliability_agent.ipynb)
- 成本与用量跟踪模式：[../observability/usage_cost_api.ipynb](../observability/usage_cost_api.ipynb)